In [1]:
import sys
from pathlib import Path

for base in (Path.cwd(), Path.cwd().parent):
    src = base / "src"
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))


# Benchmarks
Synthetic throughput and a windowed drift check.

In [2]:
import json
import time

import numpy as np
import pandas as pd
from driftalytics import SlidingWindowRunner

rng = np.random.default_rng(0)
reference = pd.DataFrame({"a": rng.normal(size=5000), "b": rng.normal(size=5000)})
results = []
for n in [1_000, 10_000, 100_000]:
    batch = pd.DataFrame({"a": rng.normal(size=n), "b": rng.normal(size=n)})
    runner = SlidingWindowRunner(
        reference=reference,
        monitors=["psi", "ks", "missing_rate"],
        window_size=1000,
    )
    start = time.perf_counter()
    report = runner.run(batch)
    elapsed = time.perf_counter() - start
    results.append(
        {
            "n": n,
            "seconds": round(elapsed, 4),
            "rows_per_sec": round(n / elapsed, 1),
            "alerts": len(report.alerts),
        }
    )
print(json.dumps(results, indent=2))

[
  {
    "n": 1000,
    "seconds": 0.0044,
    "rows_per_sec": 226577.5,
    "alerts": 0
  },
  {
    "n": 10000,
    "seconds": 0.0157,
    "rows_per_sec": 635994.8,
    "alerts": 0
  },
  {
    "n": 100000,
    "seconds": 0.0151,
    "rows_per_sec": 6634775.5,
    "alerts": 0
  }
]


In [3]:
corrupted = pd.DataFrame(
    {"a": rng.normal(loc=1.5, size=10_000), "b": rng.normal(size=10_000)}
)
runner = SlidingWindowRunner(
    reference=reference,
    monitors=["psi", "ks", "missing_rate", "drift_magnitude"],
    window_size=2_000,
    stride=2_000,
    sliding=True,
    include_partial=False,
)
report = runner.run(corrupted)
print(
    f"windows={report.n_windows} alerts={len(report.alerts)} "
    f"analyzed={report.n_rows_analyzed}"
)

windows=5 alerts=15 analyzed=10000
